In [1]:
from openai import OpenAI

In [5]:
oai_client = OpenAI(
    base_url="http://localhost:6578/v1",
    api_key="skdummy",
)

In [3]:
example_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of the United States?"},
]

In [7]:
oai_client.chat.completions.create(
    model = "meta-llama/Llama-3.1-8B-Instruct",
    messages=example_messages,
    temperature=0.0,
    max_tokens=100,
)

ChatCompletion(id='chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of the United States is Washington, D.C. (short for District of Columbia).', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[]), stop_reason=None)], created=1739596761, model='meta-llama/Llama-3.1-8B-Instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=20, prompt_tokens=50, total_tokens=70, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None)

## VLLM 怎么走的


### 1. _preprocess_chat 接到 request

```
messages=[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the capital of the United States?', 'role': 'user'}] model='meta-llama/Llama-3.1-8B-Instruct' frequency_penalty=0.0 logit_bias=None logprobs=False top_logprobs=0 max_tokens=100 max_completion_tokens=None n=1 presence_penalty=0.0 response_format=None seed=None stop=[] stream=False stream_options=None temperature=0.0 top_p=None tools=None tool_choice='none' parallel_tool_calls=False user=None best_of=None use_beam_search=False top_k=None min_p=None repetition_penalty=None length_penalty=1.0 stop_token_ids=[] include_stop_str_in_output=False ignore_eos=False min_tokens=0 skip_special_tokens=True spaces_between_special_tokens=True truncate_prompt_tokens=None prompt_logprobs=None echo=False add_generation_prompt=True continue_final_message=False add_special_tokens=False documents=None chat_template=None chat_template_kwargs=None guided_json=None guided_regex=None guided_choice=None guided_grammar=None guided_decoding_backend=None guided_whitespace_pattern=None priority=0 request_id='77c84dab6bf746dfbeb14f02e61c6e34' logits_processors=None

```

### 2. 送给hf tokenizer apply chat template 的输入

the input for tokenizer.apply_chat_template is:
```
conversation: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'What is the capital of the United States?'}]
chat_template: None
tokenize: False
Other kwargs: {'add_generation_prompt': True, 'continue_final_message': False, 'tools': None, 'documents': None}
```

输出是

```
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

```

### tokenize 

它是在外面tokenize，检查最大长度，之后再送给llm engine

llm engine的输入， prompt_token_ids`encoded = tokenizer(prompt, add_special_tokens=add_special_tokens)`的input ids

```
{'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', 'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}
```

### 送给LLM engine

engine_prompt:  {'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}


### 输出

Final result: RequestOutput(request_id=chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f, prompt=None, prompt_token_ids=[128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='The capital of the United States is Washington, D.C. (short for District of Columbia).', token_ids=(791, 6864, 315, 279, 3723, 4273, 374, 6652, 11, 423, 732, 13, 320, 8846, 369, 11182, 315, 19326, 570, 128009), cumulative_logprob=None, logprobs=None, finish_reason=stop, stop_reason=None)], finished=True, metrics=RequestMetrics(arrival_time=1739596761.045435, last_token_time=1739596761.5108209, first_scheduled_time=1739596761.0466323, first_token_time=1739596761.081663, time_in_queue=0.0011973381042480469, finished_time=1739596761.5347214, scheduler_time=0.001975471619516611, model_forward_time=None, model_execute_time=None), lora_request=None, num_cached_tokens=0, multi_modal_placeholders={})



In [8]:
import os

os.environ["HF_HOME"] = os.environ["MY_HF_HOME"]

In [9]:
from transformers import AutoTokenizer

tokenzier = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [17]:

# tokenize: False
# Other kwargs: {'add_generation_prompt': True, 'continue_final_message': False, 'tools': None, 'documents': None}
def tokenize_example_messages(example_messages):
    prompt = tokenzier.apply_chat_template(
        conversation=example_messages,
        chat_template=None,
        tokenize=False,
        add_generation_prompt=True,
        continue_final_message=False,
        tools=None,
        documents=None,
    )
    encoded = tokenzier(prompt, add_special_tokens=False)
    return encoded['input_ids']

In [12]:
example_messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is the capital of the United States?'}]

In [18]:
input_ids = tokenize_example_messages(example_messages)
print(input_ids)

[128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]


<details>
Getting chat completion for request: messages=[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the capital of the United States?', 'role': 'user'}] model='meta-llama/Llama-3.1-8B-Instruct' frequency_penalty=0.0 logit_bias=None logprobs=False top_logprobs=0 max_tokens=100 max_completion_tokens=None n=1 presence_penalty=0.0 response_format=None seed=None stop=[] stream=False stream_options=None temperature=0.0 top_p=None tools=None tool_choice='none' parallel_tool_calls=False user=None best_of=None use_beam_search=False top_k=None min_p=None repetition_penalty=None length_penalty=1.0 stop_token_ids=[] include_stop_str_in_output=False ignore_eos=False min_tokens=0 skip_special_tokens=True spaces_between_special_tokens=True truncate_prompt_tokens=None prompt_logprobs=None echo=False add_generation_prompt=True continue_final_message=False add_special_tokens=False documents=None chat_template=None chat_template_kwargs=None guided_json=None guided_regex=None guided_choice=None guided_grammar=None guided_decoding_backend=None guided_whitespace_pattern=None priority=0 request_id='f1e2559ae5df46a18fe76f9cdd79936f' logits_processors=None
Raw request: <starlette.requests.Request object at 0x7fea90d0de10>
Got tokenizer: CachedPreTrainedTokenizerFast(name_or_path='meta-llama/Llama-3.1-8B-Instruct', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='left', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={略}
)
this is _preprocess_chat
request:  messages=[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the capital of the United States?', 'role': 'user'}] model='meta-llama/Llama-3.1-8B-Instruct' frequency_penalty=0.0 logit_bias=None logprobs=False top_logprobs=0 max_tokens=100 max_completion_tokens=None n=1 presence_penalty=0.0 response_format=None seed=None stop=[] stream=False stream_options=None temperature=0.0 top_p=None tools=None tool_choice='none' parallel_tool_calls=False user=None best_of=None use_beam_search=False top_k=None min_p=None repetition_penalty=None length_penalty=1.0 stop_token_ids=[] include_stop_str_in_output=False ignore_eos=False min_tokens=0 skip_special_tokens=True spaces_between_special_tokens=True truncate_prompt_tokens=None prompt_logprobs=None echo=False add_generation_prompt=True continue_final_message=False add_special_tokens=False documents=None chat_template=None chat_template_kwargs=None guided_json=None guided_regex=None guided_choice=None guided_grammar=None guided_decoding_backend=None guided_whitespace_pattern=None priority=0 request_id='f1e2559ae5df46a18fe76f9cdd79936f' logits_processors=None
INFO 02-15 00:19:21 chat_utils.py:333] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
the input for tokenizer.apply_chat_template is:
conversation: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'What is the capital of the United States?'}]
chat_template: None
tokenize: False
Other kwargs: {'add_generation_prompt': True, 'continue_final_message': False, 'tools': None, 'documents': None}
request_prompt:  <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



_normalize_prompt_text_to_input, this is the real tokenize, input:
request:  messages=[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the capital of the United States?', 'role': 'user'}] model='meta-llama/Llama-3.1-8B-Instruct' frequency_penalty=0.0 logit_bias=None logprobs=False top_logprobs=0 max_tokens=100 max_completion_tokens=None n=1 presence_penalty=0.0 response_format=None seed=None stop=[] stream=False stream_options=None temperature=0.0 top_p=None tools=None tool_choice='none' parallel_tool_calls=False user=None best_of=None use_beam_search=False top_k=None min_p=None repetition_penalty=None length_penalty=1.0 stop_token_ids=[] include_stop_str_in_output=False ignore_eos=False min_tokens=0 skip_special_tokens=True spaces_between_special_tokens=True truncate_prompt_tokens=None prompt_logprobs=None echo=False add_generation_prompt=True continue_final_message=False add_special_tokens=False documents=None chat_template=None chat_template_kwargs=None guided_json=None guided_regex=None guided_choice=None guided_grammar=None guided_decoding_backend=None guided_whitespace_pattern=None priority=0 request_id='f1e2559ae5df46a18fe76f9cdd79936f' logits_processors=None
tokenizer:  CachedPreTrainedTokenizerFast(name_or_path='meta-llama/Llama-3.1-8B-Instruct', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='left', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={略}
)
prompt:  <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


truncate_prompt_tokens:  None
add_special_tokens:  False

Here we check the max length and pack the input
{'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', 'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}
prompt_inputs after _tokenize_prompt_input_async: 
 {'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', 'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}
LAST LINE OF _preprocess_chat
conversation:  [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'What is the capital of the United States?'}]
prompt_inputs:  {'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', 'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}
engine_prompt:  {'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}
 SEEMs like Engine prompts are what we need: [{'prompt_token_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271]}]
Request ID: chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f
Request metadata: request_id='chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f' final_usage_info=None
INFO 02-15 00:19:21 logger.py:37] Received request chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f: prompt: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the capital of the United States?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.0, top_p=1.0, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=100, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None), prompt_token_ids: None, lora_request: None, prompt_adapter_request: None.
INFO 02-15 00:19:21 engine.py:267] Added request chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f.
Final result: RequestOutput(request_id=chatcmpl-f1e2559ae5df46a18fe76f9cdd79936f, prompt=None, prompt_token_ids=[128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 279, 3723, 4273, 30, 128009, 128006, 78191, 128007, 271], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='The capital of the United States is Washington, D.C. (short for District of Columbia).', token_ids=(791, 6864, 315, 279, 3723, 4273, 374, 6652, 11, 423, 732, 13, 320, 8846, 369, 11182, 315, 19326, 570, 128009), cumulative_logprob=None, logprobs=None, finish_reason=stop, stop_reason=None)], finished=True, metrics=RequestMetrics(arrival_time=1739596761.045435, last_token_time=1739596761.5108209, first_scheduled_time=1739596761.0466323, first_token_time=1739596761.081663, time_in_queue=0.0011973381042480469, finished_time=1739596761.5347214, scheduler_time=0.001975471619516611, model_forward_time=None, model_execute_time=None), lora_request=None, num_cached_tokens=0, multi_modal_placeholders={})
INFO:     ::1:33078 - "POST /v1/chat/completions HTTP/1.1" 200 OK
</details>